In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import gc
gc.collect()

In [ ]:
# 지수 -> 소수점(2) 표기
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
gc.collect()

# 전체

In [ ]:
# chunk_iter = pd.read_csv("../Tableau/mart_total_final_oct.csv", chunksize=1000000)
chunk_iter = pd.read_csv("../Tableau/mart_total_final_nov.csv", chunksize=500000)
chunks = []

for chunk in chunk_iter:
    chunks.append(chunk)
    
df = pd.concat(chunks, axis=0)

del chunks
gc.collect()

In [ ]:
df.shape

In [ ]:
df.info

In [ ]:
df.columns

## 퍼널 분석 (2_전체)

- 메인 퍼널 전환 상관분석
- 단계별 병목 구간 특정 및 이탈 주요 영향 변수 탐색

In [ ]:
df['funnel_stage'].unique()
df['funnel_stage'].value_counts()

In [ ]:
# 전체 유저 중 몇명이 다음 구간으로 넘어갔는가 

# funnel 구간
funnel = df[['view_yn', 'cart_yn', 'purchase_yn']].sum().reset_index()
funnel.columns = ['stage', 'count']
funnel['stage'] = ['View', 'Cart', 'Purchase']
funnel['cvr'] = (funnel['count'] / funnel['count'].shift(1) * 100).fillna(100)

plt.figure(figsize=(10, 6))
sns.barplot(data=funnel, x='stage', y='count')
plt.title('병목구간 탐색을 위한 퍼널 구간 시각화')
plt.show()

In [ ]:
# 유저행동들 (특징)
cols = ['purchase_yn', 'price', 'is_weekend', 'brand_switcher_flag', 'session_duration_sec', 'time_to_purchase_sec']

# purchase_count : 총 주문/구매 건수 (price의 count)
# revenue : 총 매출액 (price의 sum)
# avg_price (SQL) : ‘브랜드 별’ 평균 상품 가격 → ROUND(AVG(price), 2) - 안 쓰기로 

# 구매 여부
# df['is_purchased'] = (df['purchase_yn'] > 0).astype(int)

sns.heatmap(df[cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")

In [ ]:
# 상관분석
# corr = df[['total_spend', 'total_views', 'total_carts', 'visit_weekend', 'is_purchased', 'purchase_count']].corr()
corr_col = ['total_spend', 'is_view', 'is_cart', 'is_weekend', 'is_purchase', 'purchase_count']

corr = df[corr_col].corr()

print(corr)

In [ ]:
# 히트맵
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('퍼널 구간 전환 및 이탈 주요 영향 변수 탐색')
plt.show()

- total_carts vs is_purchased : 어디서 빠져나가는가 (장바구니/구매)
    - 여기서 낮다 : 장바구니는 담는데 구매를 안 한다 (11월 프로모션과 연계할 가능성 有) (c-p의 병목구간)
- total_views vs is_purchased : 조회랑 구매의 관계 (이건 전처리 때 이미 본 거긴 한데) (조회/구매)
    - 낮다 : v -> p로 잘 안 이어진다 (v-c의 병목구간)
- 주말에 다녀감 : 이후 프로모션 기간이랑 대조해보면 또 인사이트 나올듯 / 예상 : 11월 블프 기간과 겹칠 것이다, 주말에는 소비가 클 것이다
- avg_price vs is_purchased : 가격/구매

- 배송이나 실제 상품 정보가 더 자세하게 나와있거나 유저 행동을 세부적으로 추측할 수 있게 수집된 데이터였으면 조금 더 볼 수 있었을 텐데 특히 배송 
- 단순 조회수랑 단순 장바구니에 담는 것만 본 거라 물론 메인 퍼널 기준이 세션+상품이라(실제 상품 기준)

In [ ]:
# 가격 차이

plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='is_purchase', y='price')
plt.title('구매 여부별 가격 분포')
plt.show()

In [ ]:
# 체류시간

plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='is_purchase', y='session_duration_sec')
plt.title('구매 여부별 체류 시간 분포')
plt.show()

In [ ]:
### SQL 데이터마트 파생변수 세부 탐색

In [ ]:
# 수치형 변수만 추출
corr_cols = [
    'purchase_yn', 'view_yn', 'cart_yn', 'price', 
    'hour', 'day_of_week', 'is_weekend', 
    'brand_missing_yn', 'is_price_error_yn', 'is_outlier_yn'
]
corr = df[corr_cols].corr()

# 히트맵
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('상관계수 히트맵', fontsize=15)
plt.show()

### decision_tier
- 구매 의사결정 시간 구간 분류(60초(1분)/5분/30분/1시간 기준)
- → ‘즉시 구매’의 세분화 

- consider_purchase_tier (즉시구매, 12, 24, 36, 36~)

In [ ]:
# decision_tier별 price 분포
### 가격 구간대의 price 분포에 따라 구매 고려 시간에 영향이 있는가 (60초(1분)/5분/30분/1시간 기준)

plt.figure(figsize=(12, 7))
sns.boxplot(data=df, x='decision_tier', y='price')
plt.title('가격 구간대의 price 분포')
plt.ylabel('price')
plt.xlabel('decision_tier')
plt.grid(axis='y', alpha=0.7)
plt.show()

### 병목 구간 세부 탐색

In [ ]:
# 이탈 및 전환 주요 영향 변수 상관분석
corr_cols = [
    'purchase_yn', 'price', 'is_weekend', 
    'brand_missing_yn', 'brand_switcher_flag', 'session_duration_sec',
    'time_to_purchase_sec' # v -> p 고민 시간
]

# 데이터에 해당 컬럼들이 있는지 확인 후 상관분석
existing_cols = [c for c in corr_cols if c in df.columns]
corr_data = df[existing_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_data, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('이탈 및 전환 영향 변수 심층 상관분석')
plt.show()

In [ ]:
# 구매자 vs 비구매자(이탈자)
plt.figure(figsize=(12, 7))
sns.boxplot(x='decision_tier', y='price', data=df)
plt.title('구매 의사결정 시간 구간별 상품 가격 분포 (이탈 요인 탐색)')
plt.show()


# 비구매자(이탈자)(0) vs 구매자(1) 비교
compare_cols = ['price', 'is_weekend', 'session_duration_sec']
for col in compare_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x='is_purchase', y=col, data=df)
    plt.title(f'구매 여부에 따른 {col} 차이')
    plt.show()

# 스마트폰

In [ ]:
sm = df[df['category_code'] == 'electronics.smartphone'].copy()

## 퍼널 분석 (2_sm)

- 메인 퍼널 전환 상관분석
- 단계별 병목 구간 특정 및 이탈 주요 영향 변수 탐색

In [ ]:
sm['funnel_stage'].unique()
sm['funnel_stage'].value_counts()

In [ ]:
# 전체 유저 중 몇명이 다음 구간으로 넘어갔는가 

# funnel 구간
funnel = sm[['view_yn', 'cart_yn', 'purchase_yn']].sum().reset_index()
funnel.columns = ['stage', 'count']
funnel['stage'] = ['View', 'Cart', 'Purchase']
funnel['cvr'] = (funnel['count'] / funnel['count'].shift(1) * 100).fillna(100)

plt.figure(figsize=(10, 6))
sns.barplot(data=funnel, x='stage', y='count')
plt.title('병목구간 탐색을 위한 퍼널 구간 시각화')
plt.show()

In [ ]:
# 유저행동들 (특징)
cols = ['purchase_yn', 'price', 'is_weekend', 'brand_switcher_flag', 'session_duration_sec', 'time_to_purchase_sec']

# purchase_count : 총 주문/구매 건수 (price의 count)
# revenue : 총 매출액 (price의 sum)
# avg_price (SQL) : ‘브랜드 별’ 평균 상품 가격 → ROUND(AVG(price), 2) - 안 쓰기로 

# 구매 여부
# sm['is_purchased'] = (sm['purchase_yn'] > 0).astype(int)

sns.heatmap(sm[cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")

In [ ]:
# 상관분석
# corr = sm[['total_spend', 'total_views', 'total_carts', 'visit_weekend', 'is_purchased', 'purchase_count']].corr()
corr_col = ['total_spend', 'is_view', 'is_cart', 'is_weekend', 'is_purchase', 'purchase_count']

corr = sm[corr_col].corr()

print(corr)

In [ ]:
# 히트맵
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('퍼널 구간 전환 및 이탈 주요 영향 변수 탐색')
plt.show()

- total_carts vs is_purchased : 어디서 빠져나가는가 (장바구니/구매)
    - 여기서 낮다 : 장바구니는 담는데 구매를 안 한다 (11월 프로모션과 연계할 가능성 有) (c-p의 병목구간)
- total_views vs is_purchased : 조회랑 구매의 관계 (이건 전처리 때 이미 본 거긴 한데) (조회/구매)
    - 낮다 : v -> p로 잘 안 이어진다 (v-c의 병목구간)
- 주말에 다녀감 : 이후 프로모션 기간이랑 대조해보면 또 인사이트 나올듯 / 예상 : 11월 블프 기간과 겹칠 것이다, 주말에는 소비가 클 것이다
- avg_price vs is_purchased : 가격/구매

- 배송이나 실제 상품 정보가 더 자세하게 나와있거나 유저 행동을 세부적으로 추측할 수 있게 수집된 데이터였으면 조금 더 볼 수 있었을 텐데 특히 배송 
- 단순 조회수랑 단순 장바구니에 담는 것만 본 거라 물론 메인 퍼널 기준이 세션+상품이라(실제 상품 기준)

In [ ]:
# 가격 차이

plt.figure(figsize=(8, 6))
sns.boxplot(data=sm, x='is_purchase', y='price')
plt.title('구매 여부별 가격 분포')
plt.show()

In [ ]:
# 체류시간

plt.figure(figsize=(8, 6))
sns.boxplot(data=sm, x='is_purchase', y='session_duration_sec')
plt.title('구매 여부별 체류 시간 분포')
plt.show()

In [ ]:
### SQL 데이터마트 파생변수 세부 탐색

In [ ]:
# 수치형 변수만 추출
corr_cols = [
    'purchase_yn', 'view_yn', 'cart_yn', 'price', 
    'hour', 'day_of_week', 'is_weekend', 
    'brand_missing_yn', 'is_price_error_yn', 'is_outlier_yn'
]
corr = sm[corr_cols].corr()

# 히트맵
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('상관계수 히트맵', fontsize=15)
plt.show()

### decision_tier
- 구매 의사결정 시간 구간 분류(60초(1분)/5분/30분/1시간 기준)
- → ‘즉시 구매’의 세분화 

- consider_purchase_tier (즉시구매, 12, 24, 36, 36~)

In [ ]:
# decision_tier별 price 분포
### 가격 구간대의 price 분포에 따라 구매 고려 시간에 영향이 있는가 (60초(1분)/5분/30분/1시간 기준)

plt.figure(figsize=(12, 7))
sns.boxplot(data=sm, x='decision_tier', y='price')
plt.title('가격 구간대의 price 분포')
plt.ylabel('price')
plt.xlabel('decision_tier')
plt.grid(axis='y', alpha=0.7)
plt.show()

### 병목 구간 세부 탐색

In [ ]:
# 이탈 및 전환 주요 영향 변수 상관분석
corr_cols = [
    'purchase_yn', 'price', 'is_weekend', 
    'brand_missing_yn', 'brand_switcher_flag', 'session_duration_sec',
    'time_to_purchase_sec' # v -> p 고민 시간
]

# 데이터에 해당 컬럼들이 있는지 확인 후 상관분석
existing_cols = [c for c in corr_cols if c in sm.columns]
corr_data = sm[existing_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_data, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('이탈 및 전환 영향 변수 심층 상관분석')
plt.show()

In [ ]:
# 구매자 vs 비구매자(이탈자)
plt.figure(figsize=(12, 7))
sns.boxplot(x='decision_tier', y='price', data=sm)
plt.title('구매 의사결정 시간 구간별 상품 가격 분포 (이탈 요인 탐색)')
plt.show()


# 비구매자(이탈자)(0) vs 구매자(1) 비교
compare_cols = ['price', 'is_weekend', 'session_duration_sec']
for col in compare_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x='is_purchase', y=col, data=sm)
    plt.title(f'구매 여부에 따른 {col} 차이')
    plt.show()